# Sports Regression - calibration_v6

Builds per-sport, per-bucket probability multipliers from 599 historical samples (FIFA World Cup 1998-2022, NBA Finals 2010-2024, NHL Stanley Cup 2010-2024, Super Bowl 2010-2024, Wimbledon mens + womens 2015-2024).

Logic mirrors `sports_regression.py` cell-by-cell.


## Imports + setup


In [ ]:
import json, os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sports_regression import (
    BUCKET_LABELS, BUCKET_EDGES, label_bucket,
    load, calibration_table, brier_per_sport,
    logistic_per_sport, multipliers,
    apply_to_current, write_calibration,
)
df = load()
print('rows:', len(df), 'sports:', sorted(df.sport.unique()))


## 3A - Calibration curves per sport

For each (sport, bucket) we count markets, count wins, compute the actual win rate, and compare to the bucket's average implied probability. `multiplier = actual_win_rate / avg_implied_prob` summarises the bias.


In [ ]:
calib = calibration_table(df)
print(calib.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


3A. Calibration curves per sport
 sport bucket  n  wins  avg_implied_prob  actual_win_rate  multiplier
  fifa    0-5 40     0            0.0223           0.0000      0.0000
  fifa   5-10 26     2            0.0732           0.0769      1.0504
  fifa  10-15 16     4            0.1231           0.2500      2.0310
  fifa  15-20  8     1            0.1836           0.1250      0.6807
  fifa  20-30  3     0            0.2548           0.0000      0.0000
  fifa  30-50  0     0               NaN              NaN         NaN
  fifa  50-75  0     0               NaN              NaN         NaN
  fifa    75+  0     0               NaN              NaN         NaN
   nba    0-5 21     2            0.0333           0.0952      2.8611
   nba   5-10 48     5            0.0695           0.1042      1.4994
   nba  10-15 23     0            0.1192           0.0000      0.0000
   nba  15-20  8     2            0.1745           0.2500      1.4328
   nba  20-30 10     1            0.2338           0.1000

In [ ]:
brier = brier_per_sport(df)
print(brier.to_string(index=False, float_format=lambda x: f'{x:.4f}'))


## 3B - Logistic regression per sport

Features: `implied_prob`, `implied_prob^2`, `log(implied_prob)`. 80/20 chronological split. Reports OOS Brier with a 200-iteration bootstrap CI95.


In [ ]:
log_models = logistic_per_sport(df)
for s, info in log_models.items():
    print(s)
    for k, v in info.items():
        print(f'  {k}: {v}')


3B. Logistic regression per sport (80/20 chronological split)
  fifa:
    n_train: 74
    n_test: 19
    coef_p: 0.015164679858499898
    coef_p_squared: -0.020664040562441862
    coef_log_p: 1.1005012948027761
    intercept: 0.3755653808391485
    brier_oos: 0.04764140171566895
    brier_oos_ci95: [0.0030194043886728836, 0.13317404030143168]
  nba:
    n_train: 96
    n_test: 25
    coef_p: 0.3775044220291547
    coef_p_squared: 0.24404057747199254
    coef_log_p: 0.644384261076987
    intercept: -0.560085520050182
    brier_oos: 0.10349491604377159
    brier_oos_ci95: [0.01896354544295806, 0.19655610594844647]
  nfl:
    n_train: 96
    n_test: 24
    coef_p: 0.08411687156888152
    coef_p_squared: 0.032410864497083955
    coef_log_p: 0.37414972853918477
    intercept: -1.1622952936332018
    brier_oos: 0.1066146761369307
    brier_oos_ci95: [0.015380185462956475, 0.20030296312980678]
  nhl:
    n_train: 96
    n_test: 25
    coef_p: 0.05125431605222538
    coef_p_squared: 0.01757489

## 3C - Derived multipliers per probability bucket

Laplace smoothing (alpha=1) shrinks each cell's actual win rate toward the global per-bucket prior. Multipliers clamped to `[0.05, 2.0]` so a 1-of-1 cell does not produce a 10x estimate.


In [ ]:
mults = multipliers(df)
for sport in sorted(mults):
    row = mults[sport]
    print(sport.ljust(10), ' '.join(f'{row[b]:7.3f}' for b in BUCKET_LABELS))


3C. Derived multipliers per bucket (Laplace alpha=1, clamped [0.05, 2.0])
sport          0-5    5-10   10-15   15-20   20-30   30-50   50-75     75+
fifa         0.419   1.057   1.606   0.801   0.533   1.000   1.000   1.000
nba          2.000   1.436   0.290   1.161   0.591   1.171   1.188   1.000
nfl          1.000   0.963   0.985   0.943   0.979   1.000   1.000   1.000
nhl          0.803   1.234   0.808   1.006   1.022   1.000   1.000   1.000
tennis       1.972   0.470   1.141   0.802   0.741   1.135   1.298   1.000



## 3D - Apply v6 multipliers to current 2026 tournament markets

Pulls live Polymarket prices via the running scanner API (`localhost:3001`). Each team's `p_market` is mapped to its bucket; multiplier produces a v6 `p_model`; `raw_edge = p_market - p_model_v6`. Positive edges are SHORT candidates, negative are LONG.


In [ ]:
current = apply_to_current(mults)
import collections
by_tour = collections.defaultdict(list)
for r in current:
    by_tour[r['tournament']].append(r)
for tour in sorted(by_tour):
    print(tour)
    rows = sorted(by_tour[tour], key=lambda x: -x['raw_edge_v6'])
    for r in rows[:15]:
        side = 'SHORT' if r['raw_edge_v6'] > 0.01 else ('LONG' if r['raw_edge_v6'] < -0.01 else 'FAIR')
        print(f"  {side:<5} p={r['p_market']:.3f} mult={r['multiplier_v6']:.3f} p_model={r['p_model_v6']:.3f} edge={r['raw_edge_v6']:+.4f}  {r['question'][:80]}")


3D. Apply v6 multipliers to current 2026 tournament markets

2026 FIFA World Cup (50 teams)
  side    p_mkt  bucket   mult   p_mdl     edge           vol  question
  SHORT   0.177   15-20  0.801   0.142  +0.0354    29,271,911  Will France win the 2026 FIFA World Cup?
  SHORT   0.174   15-20  0.801   0.140  +0.0348    23,018,812  Will Spain win the 2026 FIFA World Cup?
  SHORT   0.035     0-5  0.419   0.015  +0.0204             0  Will Africa win the 2026 FIFA World Cup?
  SHORT   0.035     0-5  0.419   0.014  +0.0201    20,853,366  Will Netherlands win the 2026 FIFA World Cup?
  SHORT   0.025     0-5  0.419   0.010  +0.0142    20,720,121  Will Norway win the 2026 FIFA World Cup?
  SHORT   0.018     0-5  0.419   0.008  +0.0108    18,453,550  Will Belgium win the 2026 FIFA World Cup?
  SHORT   0.018     0-5  0.419   0.007  +0.0102    25,064,410  Will Japan win the 2026 FIFA World Cup?
  FAIR    0.017     0-5  0.419   0.007  +0.0096    18,585,723  Will Colombia win the 2026 FIFA World Cup

## 3E - Write calibration_v6.json


In [ ]:
out = write_calibration(mults, {s: int(len(g)) for s, g in df.groupby('sport')}, log_models, calib)
print('wrote', out)


3E. Write calibration_v6.json
wrote C:\Users\alexs\OneDrive\Desktop\Homework USC\Personal Coding\Contra\ml\artifacts\calibration_v6.json
